# 🧬 Protenix Structure Prediction Example

This notebook demonstrates how to predict protein and protein-ligand complex structures using **Protenix**.

## 📖 What is Protenix?

Protenix is an open-source reimplementation of AlphaFold3 developed by ByteDance Research. It predicts structures of proteins, nucleic acids (DNA/RNA), and ligands, as well as their complexes.

### Key Features:
- **Multi-modal** -- Supports proteins, DNA, RNA, and ligands
- **Complex prediction** -- Handles protein-ligand, protein-DNA/RNA, and multi-chain complexes
- **Confidence scores** -- Provides metrics including pTM, ipTM, pLDDT, and GPDE
- **MSA integration** -- Uses ColabFold for multiple sequence alignments
- **Flexible** -- Multiple model sizes (base, mini, tiny) and configurable diffusion/recycle steps

## 📥 Imports

## 📦 Shared Data Types

### `StructurePredictionComplex`
A biological complex containing one or more molecular chains to predict together.

| Field | Type | Description |
|-------|------|-------------|
| `chains` | `List[Chain]` | Chains in the complex. Accepts strings, `Chain` objects, or dicts |

### `Chain`
A single molecular chain with optional modifications.

| Field | Type | Description |
|-------|------|-------------|
| `sequence` | `str` | Sequence (protein amino acids, DNA/RNA bases, or ligand SMILES) |
| `entity_type` | `Optional[str]` | `"protein"`, `"dna"`, `"rna"`, or `"ligand"`. Auto-inferred if `None` |
| `modifications` | `List[ChainModification]` | Post-translational or nucleotide modifications. Default: `[]` |

### `ChainModification`
A modification at a specific position in a chain, specified using CCD codes.

| Field | Type | Description |
|-------|------|-------------|
| `position` | `int` | 1-based position in the sequence |
| `modification_code` | `str` | CCD code (e.g., `"SEP"` for phosphoserine, `"TPO"` for phosphothreonine) |

### `Structure`
A predicted 3D structure with coordinates, confidence metrics, and export methods.

## 📥 Imports

In [1]:
from pathlib import Path

from proto_tools import (
    Chain,
    ProtenixConfig,
    ProtenixInput,
    StructurePredictionComplex,
    run_protenix,
)

## 🔍 Protein-Ligand Complex Prediction (MfnG Example)

Let's predict the structure of MfnG protein bound to L-tyrosine ligand.

### API Reference

**`ProtenixInput`**

| Field | Type | Description |
|-------|------|-------------|
| `complexes` | `List[StructurePredictionComplex]` | List of complexes to predict structures for |

**`ProtenixConfig`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `model_name` | `ProtenixModelName` | `"protenix_base_default_v1.0.0"` | Protenix model variant to use (base/mini/tiny variants available) |
| `seeds` | `List[int]` | `[0]` | Random seeds for structure sampling |
| `use_msa` | `bool` | `True` | Whether to generate and use MSAs for protein chains using ColabFold search |
| `num_diffusion_samples` | `int` | `5` | Number of independent structure samples per seed (only best is returned) |
| `num_diffusion_steps` | `int` | `200` | Number of denoising steps in the diffusion process |
| `num_pairformer_cycles` | `int` | `10` | Number of Pairformer recycling iterations |

**`ProtenixOutput`**

| Field | Type | Description |
|-------|------|-------------|
| `structures` | `List[Structure]` | List of predicted structures, one per input complex |

In [2]:
# MfnG protein sequence
mfng_sequence = "MVTPEGNVSLVDESLLVGVTDEDRAVRSAHQFYERLIGLWAPAVMEAAHELGVFAALAEAPADSGELARRLDCDARAMRVLLDALYAYDVIDRIHDTNGFRYLLSAEARECLLPGTLFSLVGKFMHDINVAWPAWRNLAEVVRHGARDTSGAESPNGIAQEDYESLVGGINFWAPPIVTTLSRKLRASGRSGDATASVLDVGCGTGLYSQLLLREFPRWTATGLDVERIATLANAQALRLGVEERFATRAGDFWRGGWGTGYDLVLFANIFHLQTPASAVRLMRHAAACLAPDGLVAVVDQIVDADREPKTPQDRFALLFAASMTNTGGGDAYTFQEYEEWFTAAGLQRIETLDTPMHRILLARRATEPSAVPEGQASENLYFQ"

# L-tyrosine SMILES
tyrosine_smiles = "c1cc(ccc1C[C@@H](C(=O)O)N)O"

# Create protein-ligand complex
complex = StructurePredictionComplex(
    chains=[
        Chain(sequence=mfng_sequence, entity_type="protein"),
        Chain(sequence=tyrosine_smiles, entity_type="ligand"),
    ]
)

# Create input
inputs = ProtenixInput(complexes=[complex])

# Configure Protenix
config = ProtenixConfig(
    verbose=False,
    device="cuda",  # Change to "cpu" if no GPU available
    model_name="protenix_base_default_v1.0.0",
    use_msa=True,
    num_pairformer_cycles=10,
    num_diffusion_samples=5,
    num_diffusion_steps=200,
    seeds=[0],
)

# Run prediction
result = run_protenix(inputs, config)
mfng_structure = result.structures[0]

# Print metrics
print(f"\n✅ Structure predicted!")
print(f"  Number of chains: {len(complex.chains)}")
print(f"  Protein length: {len(mfng_sequence)} residues")
print(f"  Confidence score: {mfng_structure.confidence_score:.3f}")
print(f"  Average pLDDT: {mfng_structure.avg_plddt:.3f}")
print(f"  pTM score: {mfng_structure.ptm:.3f}")
print(f"  ipTM score: {mfng_structure.iptm:.3f}")
print(f"  GPDE: {mfng_structure.metrics.get('gpde', 'N/A')} Å")


✅ Structure predicted!
  Number of chains: 2
  Protein length: 384 residues
  Confidence score: 0.937
  Average pLDDT: 0.902
  pTM score: 0.913
  ipTM score: 0.943
  GPDE: 0.46013572812080383 Å


### 🎨 Visualize MfnG-Ligand Complex

In [6]:
mfng_structure.visualize(style='cartoon', color_by='bfactor')

## 💾 Export Results

Save predicted structures for further analysis in other tools like PyMOL, ChimeraX, or VMD.

### Supported formats:
- **PDB** -- Standard protein data bank format, widely compatible
- **mmCIF** -- Modern crystallographic information file, supports larger structures

The B-factor column contains the pLDDT scores for confidence visualization.

In [ ]:
# Create output directory
output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

# Export results to pdb files
result.export(name="mfng_ligand_complex", export_path=output_dir, file_format="pdb")